In [3]:
#!/usr/bin/env Rscript

# 01_figures_macro_f1.R
# Run from the project root:
#   Rscript 01_figures_macro_f1.R
#
# Required package:
#   install.packages("tidyverse")

suppressPackageStartupMessages(library(tidyverse))

out_dir <- "paper_analysis_diagnosis_5fold_chatgpt"
dir.create(out_dir, showWarnings = FALSE)

budgets_keep <- c("10", "25", "50", "100", "250", "full")

read_result <- function(path, method, mode_col = NULL) {
  x <- read_csv(path, show_col_types = FALSE)

  if (is.null(mode_col)) {
    x %>%
      transmute(
        method = method,
        fold = as.integer(fold),
        budget = as.character(budget),
        f1_macro = as.numeric(f1_macro)
      )
  } else {
    x %>%
      transmute(
        method = recode(
          .data[[mode_col]],
          "head" = "DINO head",
          "last_block" = "DINO last block",
          "last_2_blocks" = "DINO last 2 blocks",
          "last_4_blocks" = "DINO last 4 blocks"
        ),
        fold = as.integer(fold),
        budget = as.character(budget),
        f1_macro = as.numeric(f1_macro)
      )
  }
}

dino <- read_result(
  "outputs_dino_diagnosis_multiclass_5fold/results_linear_probe.csv",
  "DINOv2 frozen"
)

mae10 <- read_result(
  "outputs_mae10k_diagnosis_multiclass_5fold/results_mae_linear_probe.csv",
  "MAE-SSL-10k"
)

mae50 <- read_result(
  "outputs_mae50k_diagnosis_multiclass_5fold/results_mae_linear_probe.csv",
  "MAE-SSL-50k"
)

mae100 <- read_result(
  "outputs_mae100k_diagnosis_multiclass_5fold/results_mae_linear_probe.csv",
  "MAE-SSL-100k"
)

partial <- read_result(
  "outputs_dino_partialft_diagnosis_multiclass_5fold/results_finetune.csv",
  method = NULL,
  mode_col = "finetune_mode"
)

results <- bind_rows(dino, mae10, mae50, mae100, partial) %>%
  filter(budget %in% budgets_keep)

summary_df <- results %>%
  group_by(method, budget) %>%
  summarise(
    mean_f1 = mean(f1_macro, na.rm = TRUE),
    sd_f1 = sd(f1_macro, na.rm = TRUE),
    .groups = "drop"
  )

write_csv(summary_df, file.path(out_dir, "macro_f1_summary.csv"))

representation_methods <- c(
  "DINOv2 frozen",
  "MAE-SSL-10k",
  "MAE-SSL-50k",
  "MAE-SSL-100k"
)

p_rep <- summary_df %>%
  filter(method %in% representation_methods) %>%
  mutate(budget = factor(budget, levels = budgets_keep)) %>%
  ggplot(aes(budget, mean_f1, group = method, linetype = method, shape = method)) +
  geom_line() +
  geom_point(size = 2.1) +
  geom_errorbar(
    aes(ymin = mean_f1 - sd_f1, ymax = mean_f1 + sd_f1),
    width = 0.1
  ) +
  labs(
    x = "Number of labeled training images",
    y = "Macro-F1",
    linetype = NULL,
    shape = NULL
  ) +
  theme_classic() +
  theme(legend.position = "bottom")

ggsave(file.path(out_dir, "figure_representation_macro_f1.pdf"), p_rep, width = 7, height = 4)
ggsave(file.path(out_dir, "figure_representation_macro_f1.png"), p_rep, width = 7, height = 4, dpi = 300)

finetuning_methods <- c(
  "DINOv2 frozen",
  "DINO head",
  "DINO last block",
  "DINO last 2 blocks",
  "DINO last 4 blocks"
)

p_ft <- summary_df %>%
  filter(method %in% finetuning_methods) %>%
  mutate(budget = factor(budget, levels = budgets_keep)) %>%
  ggplot(aes(budget, mean_f1, group = method, linetype = method, shape = method)) +
  geom_line() +
  geom_point(size = 2.1) +
  geom_errorbar(
    aes(ymin = mean_f1 - sd_f1, ymax = mean_f1 + sd_f1),
    width = 0.1
  ) +
  labs(
    x = "Number of labeled training images",
    y = "Macro-F1",
    linetype = NULL,
    shape = NULL
  ) +
  theme_classic() +
  theme(legend.position = "bottom")

ggsave(file.path(out_dir, "figure_finetuning_macro_f1.pdf"), p_ft, width = 7, height = 4)
ggsave(file.path(out_dir, "figure_finetuning_macro_f1.png"), p_ft, width = 7, height = 4, dpi = 300)

message("Done: ", out_dir)

Done: paper_analysis_diagnosis_5fold_chatgpt



In [7]:
#!/usr/bin/env Rscript

# 02_full_table_f1_qwk.R
# Run from the project root:
#   Rscript 02_full_table_f1_qwk.R
#
# Required package:
#   install.packages("tidyverse")

suppressPackageStartupMessages(library(tidyverse))

out_dir <- "paper_analysis_diagnosis_5fold_chatgpt"
dir.create(out_dir, showWarnings = FALSE)

macro_f1 <- function(y_true, y_pred) {
  classes <- sort(unique(c(y_true, y_pred)))
  mean(sapply(classes, function(cls) {
    tp <- sum(y_true == cls & y_pred == cls)
    fp <- sum(y_true != cls & y_pred == cls)
    fn <- sum(y_true == cls & y_pred != cls)
    precision <- ifelse(tp + fp == 0, 0, tp / (tp + fp))
    recall <- ifelse(tp + fn == 0, 0, tp / (tp + fn))
    ifelse(precision + recall == 0, 0, 2 * precision * recall / (precision + recall))
  }))
}

qwk <- function(y_true, y_pred) {
  labels <- sort(unique(c(y_true, y_pred)))
  k <- length(labels)
  true_idx <- match(y_true, labels)
  pred_idx <- match(y_pred, labels)

  observed <- table(
    factor(true_idx, levels = seq_len(k)),
    factor(pred_idx, levels = seq_len(k))
  )
  observed <- observed / sum(observed)

  expected <- outer(rowSums(observed), colSums(observed))
  weights <- outer(
    seq_len(k),
    seq_len(k),
    function(i, j) ((i - j)^2) / ((k - 1)^2)
  )

  1 - sum(weights * observed) / sum(weights * expected)
}

standardize <- function(df, method, default_budget = "full") {
  if (!"budget" %in% names(df)) df$budget <- default_budget

  df %>%
    transmute(
      method = method,
      fold = as.integer(fold),
      budget = as.character(budget),
      y_true = as.integer(y_true),
      y_pred = as.integer(y_pred)
    ) %>%
    filter(budget == "full")
}

# DINO frozen: only the five full-budget files.
dino_files <- sort(list.files(
  "outputs_dino_diagnosis_multiclass_5fold/predictions",
  pattern = "^fold[0-9]+_budgetfull_predictions\\.csv$",
  full.names = TRUE
))

dino <- map_dfr(
  dino_files,
  ~ standardize(read_csv(.x, show_col_types = FALSE), "DINOv2 frozen")
)

mae10 <- read_csv(
  "outputs_mae10k_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  show_col_types = FALSE,
  col_types = cols(.default = col_character())
) %>% standardize("MAE-SSL-10k")

mae50 <- read_csv(
  "outputs_mae50k_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  show_col_types = FALSE,
  col_types = cols(.default = col_character())
) %>% standardize("MAE-SSL-50k")

mae100 <- read_csv(
  "outputs_mae100k_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  show_col_types = FALSE,
  col_types = cols(.default = col_character())
) %>% standardize("MAE-SSL-100k")

partial_raw <- read_csv(
  "outputs_dino_partialft_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  show_col_types = FALSE,
  col_types = cols(.default = col_character())
)

partial <- partial_raw %>%
  filter(budget == "full") %>%
  mutate(
    method = recode(
      finetune_mode,
      "head" = "DINO head",
      "last_block" = "DINO last block",
      "last_2_blocks" = "DINO last 2 blocks",
      "last_4_blocks" = "DINO last 4 blocks"
    )
  ) %>%
  transmute(
    method,
    fold = as.integer(fold),
    budget = "full",
    y_true = as.integer(y_true),
    y_pred = as.integer(y_pred)
  )

resnet_none <- read_csv(
  "outputs_resnet50_scratch_diagnosis_multiclass_5fold/predictions_all_folds_none.csv",
  show_col_types = FALSE
) %>% standardize("ResNet50 scratch")

resnet_imagenet <- read_csv(
  "outputs_resnet50_scratch_diagnosis_multiclass_5fold/predictions_all_folds_imagenet.csv",
  show_col_types = FALSE
) %>% standardize("ResNet50 ImageNet")

all_predictions <- bind_rows(
  dino,
  mae10,
  mae50,
  mae100,
  partial,
  resnet_none,
  resnet_imagenet
)

by_fold <- all_predictions %>%
  group_by(method, fold) %>%
  summarise(
    macro_f1 = macro_f1(y_true, y_pred),
    qwk = qwk(y_true, y_pred),
    .groups = "drop"
  )

write_csv(by_fold, file.path(out_dir, "full_data_metrics_by_fold.csv"))

summary_table <- by_fold %>%
  group_by(method) %>%
  summarise(
    macro_f1_mean = mean(macro_f1),
    macro_f1_sd = sd(macro_f1),
    qwk_mean = mean(qwk),
    qwk_sd = sd(qwk),
    .groups = "drop"
  ) %>%
  transmute(
    Method = method,
    `Macro-F1` = sprintf("%.3f ± %.3f", macro_f1_mean, macro_f1_sd),
    QWK = sprintf("%.3f ± %.3f", qwk_mean, qwk_sd)
  )

write_csv(summary_table, file.path(out_dir, "table_full_data_f1_qwk.csv"))
print(summary_table)

message("Done: ", out_dir)

# A tibble: 10 × 3
   Method             `Macro-F1`    QWK          
   <chr>              <chr>         <chr>        
 1 DINO head          0.570 ± 0.025 0.553 ± 0.066
 2 DINO last 2 blocks 0.618 ± 0.041 0.635 ± 0.070
 3 DINO last 4 blocks 0.683 ± 0.034 0.704 ± 0.047
 4 DINO last block    0.595 ± 0.039 0.576 ± 0.058
 5 DINOv2 frozen      0.563 ± 0.041 0.536 ± 0.049
 6 MAE-SSL-100k       0.519 ± 0.027 0.511 ± 0.070
 7 MAE-SSL-10k        0.522 ± 0.029 0.474 ± 0.067
 8 MAE-SSL-50k        0.526 ± 0.014 0.494 ± 0.056
 9 ResNet50 ImageNet  0.657 ± 0.032 0.654 ± 0.047
10 ResNet50 scratch   0.564 ± 0.059 0.570 ± 0.065


Done: paper_analysis_diagnosis_5fold_chatgpt



In [8]:
#!/usr/bin/env Rscript

# 03_glmm_full_data.R
# Run after 02_full_table_f1_qwk.R:
#   Rscript 03_glmm_full_data.R
#
# Required packages:
#   install.packages(c("tidyverse", "lme4", "emmeans"))

suppressPackageStartupMessages({
  library(tidyverse)
  library(lme4)
  library(emmeans)
})

out_dir <- "paper_analysis_diagnosis_5fold_chatgpt"

# Rebuild only the full-budget patient-level predictions.
read_full <- function(path, method, budget_col = TRUE) {
  x <- read_csv(
    path,
    show_col_types = FALSE,
    col_types = cols(.default = col_character())
  )

  if (budget_col) x <- x %>% filter(budget == "full")

  x %>%
    transmute(
      method = method,
      fold = factor(fold),
      participant_id = factor(participant_id),
      y_true = as.integer(y_true),
      y_pred = as.integer(y_pred),
      correct = as.integer(y_true == y_pred)
    )
}

dino_files <- sort(list.files(
  "outputs_dino_diagnosis_multiclass_5fold/predictions",
  pattern = "^fold[0-9]+_budgetfull_predictions\\.csv$",
  full.names = TRUE
))

dino <- map_dfr(dino_files, ~ read_full(.x, "DINOv2 frozen", budget_col = FALSE))

mae10 <- read_full(
  "outputs_mae10k_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  "MAE-SSL-10k"
)

mae50 <- read_full(
  "outputs_mae50k_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  "MAE-SSL-50k"
)

mae100 <- read_full(
  "outputs_mae100k_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  "MAE-SSL-100k"
)

resnet_none <- read_full(
  "outputs_resnet50_scratch_diagnosis_multiclass_5fold/predictions_all_folds_none.csv",
  "ResNet50 scratch",
  budget_col = FALSE
)

resnet_imagenet <- read_full(
  "outputs_resnet50_scratch_diagnosis_multiclass_5fold/predictions_all_folds_imagenet.csv",
  "ResNet50 ImageNet",
  budget_col = FALSE
)

partial_raw <- read_csv(
  "outputs_dino_partialft_diagnosis_multiclass_5fold/predictions_all_folds.csv",
  show_col_types = FALSE,
  col_types = cols(.default = col_character())
) %>%
  filter(budget == "full") %>%
  mutate(
    method = recode(
      finetune_mode,
      "head" = "DINO head",
      "last_block" = "DINO last block",
      "last_2_blocks" = "DINO last 2 blocks",
      "last_4_blocks" = "DINO last 4 blocks"
    )
  ) %>%
  transmute(
    method,
    fold = factor(fold),
    participant_id = factor(participant_id),
    y_true = as.integer(y_true),
    y_pred = as.integer(y_pred),
    correct = as.integer(y_true == y_pred)
  )

model_data <- bind_rows(
  dino,
  mae10,
  mae50,
  mae100,
  resnet_none,
  resnet_imagenet,
  partial_raw
) %>%
  mutate(method = factor(method))

model <- glmer(
  correct ~ method + (1 | participant_id) + (1 | fold),
  data = model_data,
  family = binomial,
  nAGQ = 0,
  control = glmerControl(
    optimizer = "bobyqa",
    optCtrl = list(maxfun = 100000)
  )
)

saveRDS(model, file.path(out_dir, "glmm_full_data.rds"))

fixed_effects <- as.data.frame(summary(model)$coefficients) %>%
  rownames_to_column("term")

write_csv(fixed_effects, file.path(out_dir, "glmm_fixed_effects.csv"))

pairwise <- emmeans(
  model,
  pairwise ~ method,
  type = "response",
  adjust = "tukey"
)

write_csv(
  as.data.frame(pairwise$contrasts),
  file.path(out_dir, "glmm_pairwise_tukey.csv")
)

message("Done: ", out_dir)

Done: paper_analysis_diagnosis_5fold_chatgpt



In [9]:
#!/usr/bin/env Rscript

# duy_external_validation.R
# Separate analysis for the external Duy dataset.
#
# Run from the project root:
#   Rscript duy_external_validation.R
#
# Required package:
#   install.packages("tidyverse")
#
# Edit ONLY the "files" table below with the real prediction CSV paths.
# Each CSV must contain at least: y_true, y_pred
# Optional columns: fold, participant_id

suppressPackageStartupMessages(library(tidyverse))

out_dir <- "paper_analysis_duy"
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# -------------------------------------------------------------------------
# EDIT ONLY THIS BLOCK
# -------------------------------------------------------------------------

files <- tribble(
  ~method,               ~path,
  "DINOv2 frozen",       "PATH/TO/dino_duy_predictions.csv",
  "MAE-SSL-100k",        "PATH/TO/mae100k_duy_predictions.csv",
  "ResNet50 ImageNet",   "PATH/TO/resnet_imagenet_duy_predictions.csv",
  "DINO last 4 blocks",  "PATH/TO/dino_last4_duy_predictions.csv"
)

# Remove any row above for a method that was not evaluated on Duy.

# -------------------------------------------------------------------------
# METRICS
# -------------------------------------------------------------------------

macro_f1 <- function(y_true, y_pred) {
  classes <- sort(unique(c(y_true, y_pred)))

  mean(sapply(classes, function(cls) {
    tp <- sum(y_true == cls & y_pred == cls)
    fp <- sum(y_true != cls & y_pred == cls)
    fn <- sum(y_true == cls & y_pred != cls)

    precision <- ifelse(tp + fp == 0, 0, tp / (tp + fp))
    recall <- ifelse(tp + fn == 0, 0, tp / (tp + fn))

    ifelse(
      precision + recall == 0,
      0,
      2 * precision * recall / (precision + recall)
    )
  }))
}

qwk <- function(y_true, y_pred) {
  labels <- sort(unique(c(y_true, y_pred)))
  k <- length(labels)

  if (k < 2) {
    return(NA_real_)
  }

  true_idx <- match(y_true, labels)
  pred_idx <- match(y_pred, labels)

  observed <- table(
    factor(true_idx, levels = seq_len(k)),
    factor(pred_idx, levels = seq_len(k))
  )
  observed <- observed / sum(observed)

  expected <- outer(rowSums(observed), colSums(observed))
  weights <- outer(
    seq_len(k),
    seq_len(k),
    function(i, j) ((i - j)^2) / ((k - 1)^2)
  )

  expected_disagreement <- sum(weights * expected)

  if (expected_disagreement == 0) {
    return(NA_real_)
  }

  1 - sum(weights * observed) / expected_disagreement
}

# -------------------------------------------------------------------------
# LOAD PREDICTIONS
# -------------------------------------------------------------------------

read_predictions <- function(method, path) {
  if (!file.exists(path)) {
    stop("Missing file for ", method, ": ", path)
  }

  x <- read_csv(
    path,
    show_col_types = FALSE,
    col_types = cols(.default = col_character())
  )

  required <- c("y_true", "y_pred")
  missing <- setdiff(required, names(x))

  if (length(missing) > 0) {
    stop(
      method,
      " is missing columns: ",
      paste(missing, collapse = ", ")
    )
  }

  if (!"fold" %in% names(x)) {
    x$fold <- "1"
  }

  x %>%
    transmute(
      method = method,
      fold = as.integer(fold),
      y_true = as.integer(y_true),
      y_pred = as.integer(y_pred)
    )
}

predictions <- pmap_dfr(files, read_predictions)

write_csv(
  predictions,
  file.path(out_dir, "duy_predictions_harmonized.csv")
)

# -------------------------------------------------------------------------
# RESULTS
# -------------------------------------------------------------------------

results_by_fold <- predictions %>%
  group_by(method, fold) %>%
  summarise(
    n = n(),
    macro_f1 = macro_f1(y_true, y_pred),
    qwk = qwk(y_true, y_pred),
    .groups = "drop"
  )

write_csv(
  results_by_fold,
  file.path(out_dir, "duy_metrics_by_fold.csv")
)

results_summary <- results_by_fold %>%
  group_by(method) %>%
  summarise(
    n_folds = n(),
    macro_f1_mean = mean(macro_f1, na.rm = TRUE),
    macro_f1_sd = ifelse(n() > 1, sd(macro_f1, na.rm = TRUE), NA_real_),
    qwk_mean = mean(qwk, na.rm = TRUE),
    qwk_sd = ifelse(n() > 1, sd(qwk, na.rm = TRUE), NA_real_),
    .groups = "drop"
  )

write_csv(
  results_summary,
  file.path(out_dir, "duy_metrics_summary.csv")
)

paper_table <- results_summary %>%
  transmute(
    Method = method,
    `Macro-F1` = ifelse(
      n_folds > 1,
      sprintf("%.3f ± %.3f", macro_f1_mean, macro_f1_sd),
      sprintf("%.3f", macro_f1_mean)
    ),
    QWK = ifelse(
      n_folds > 1,
      sprintf("%.3f ± %.3f", qwk_mean, qwk_sd),
      sprintf("%.3f", qwk_mean)
    )
  ) %>%
  arrange(desc(`Macro-F1`))

write_csv(
  paper_table,
  file.path(out_dir, "duy_table_external_validation.csv")
)

print(paper_table)

# Optional compact figure.
plot_data <- results_summary %>%
  select(method, Macro_F1 = macro_f1_mean, QWK = qwk_mean) %>%
  pivot_longer(
    cols = c(Macro_F1, QWK),
    names_to = "metric",
    values_to = "value"
  ) %>%
  mutate(
    metric = recode(metric, "Macro_F1" = "Macro-F1")
  )

p <- ggplot(
  plot_data,
  aes(x = reorder(method, value), y = value, shape = metric)
) +
  geom_point(size = 3) +
  coord_flip() +
  facet_wrap(~ metric, scales = "free_y") +
  labs(
    x = NULL,
    y = "Performance",
    shape = NULL
  ) +
  theme_classic() +
  theme(legend.position = "none")

ggsave(
  file.path(out_dir, "duy_external_validation.pdf"),
  p,
  width = 6.5,
  height = 3.5
)

ggsave(
  file.path(out_dir, "duy_external_validation.png"),
  p,
  width = 6.5,
  height = 3.5,
  dpi = 300
)

message("Done. Results saved in: ", out_dir)

ERROR: [1m[33mError[39m in `pmap()`:[22m
[1m[22m[36mℹ[39m In index: 1.
[1mCaused by error in `.f()`:[22m
[33m![39m Missing file for DINOv2 frozen: PATH/TO/dino_duy_predictions.csv
